# Can we predict 30-day readmission?
Target: `readmit_30d`. Rows are hospital stays; the same person appears in many rows, so the key design choice is to **split by person** (GroupKFold) – otherwise the model can memorise people and the scores are optimistic.

Plan: data checks → baseline → models with person-grouped CV → held-out test → calibration / usefulness → conclusion.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',150)
df=pd.read_csv('data.csv'); dd=pd.read_csv('columns.csv')
df.shape, df.isna().sum().sum()

## 1. Data checks

In [ ]:
print('persons:',df.person_key.nunique(),' stays/person:',df.groupby('person_key').size().describe()[['min','mean','max']].round(1).to_dict())
print('duplicate rows:',df.duplicated().sum(),' dup (person,visit):',df.duplicated(['person_key','visit_number']).sum())
print('readmission rate:',df.readmit_30d.mean().round(3))
df.describe().T.round(2)

In [ ]:
# Is readmission clustered within people?
pr=df.groupby('person_key').readmit_30d.agg(['mean','size'])
print('between-person SD of rate:',pr['mean'].std().round(3),' vs binomial expectation ~',np.sqrt(.2*.8/pr['size'].mean()).round(3))

In [ ]:
num=['visit_number','birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b','postcode_area']
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,c in zip(ax.ravel(),num):
    q=pd.qcut(df[c],10,duplicates='drop'); r=df.groupby(q,observed=True).readmit_30d.mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(df.readmit_30d.mean(),ls='--',c='grey'); a.set_title(c); a.set_xlabel('decile'); a.set_ylabel('readmit rate')
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())

## 2. Features
`person_key` is an identifier and is **not** used as a feature. `postcode_area` is a numeric code, i.e. a category: one-hot for the logistic model, raw for trees. There is no admission date, so `birth_year` is used directly rather than age.

In [ ]:
X=df.drop(columns=['person_key','readmit_30d']).copy()
X['sex']=(X.sex=='M').astype(int)
y=df.readmit_30d.values; g=df.person_key.values
# Hold out 20% of PEOPLE as a final test set
tr,te=next(GroupShuffleSplit(1,test_size=.2,random_state=0).split(X,y,g))
Xtr,ytr,gtr,Xte,yte=X.iloc[tr],y[tr],g[tr],X.iloc[te],y[te]
print(len(tr),len(te),ytr.mean().round(3),yte.mean().round(3),'overlap of people:',len(set(gtr)&set(g[te])))

## 3. Models, person-grouped 5-fold CV on the training people

In [ ]:
lin_cols=[c for c in X.columns if c!='postcode_area']
lr=make_pipeline(ColumnTransformer([('n',StandardScaler(),lin_cols),('p',OneHotEncoder(handle_unknown='ignore'),['postcode_area'])]),LogisticRegression(max_iter=2000,C=.5))
lr_np=make_pipeline(ColumnTransformer([('n',StandardScaler(),lin_cols)]),LogisticRegression(max_iter=2000))
hgb=HistGradientBoostingClassifier(max_depth=3,learning_rate=.04,max_iter=250,min_samples_leaf=60,l2_regularization=5,random_state=0)
models={'baseline (prevalence)':DummyClassifier(strategy='prior'),'logistic':lr,'logistic, no postcode':lr_np,'grad. boosting':hgb}
cv=GroupKFold(5); rows=[]
for n,m in models.items():
    p=cross_val_predict(m,Xtr,ytr,groups=gtr,cv=cv,method='predict_proba')[:,1]
    fa=[roc_auc_score(ytr[i],p[i]) for _,i in cv.split(Xtr,ytr,gtr)]
    rows.append(dict(model=n,AUC=roc_auc_score(ytr,p),AUC_fold_sd=np.std(fa),PR_AUC=average_precision_score(ytr,p),Brier=brier_score_loss(ytr,p)))
pd.DataFrame(rows).round(3)

## 4. Final held-out test (people never seen in training), bootstrap CIs resampling people

In [ ]:
rng=np.random.default_rng(0)
fitted={n:models[n].fit(Xtr,ytr) for n in('logistic','grad. boosting')}
pt={n:m.predict_proba(Xte)[:,1] for n,m in fitted.items()}
gte=g[te]; ppl=np.unique(gte); idx={p:np.where(gte==p)[0] for p in ppl}
def boot(p,B=500):
    out=[]
    for _ in range(B):
        i=np.concatenate([idx[k] for k in rng.choice(ppl,len(ppl))]); out.append(roc_auc_score(yte[i],p[i]))
    return np.percentile(out,[2.5,97.5])
for n,p in pt.items():
    lo,hi=boot(p); print(f'{n:15s} test AUC {roc_auc_score(yte,p):.3f}  95% CI [{lo:.3f}, {hi:.3f}]  PR-AUC {average_precision_score(yte,p):.3f} (prevalence {yte.mean():.3f})')

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
for n,p in pt.items():
    f,t,_=roc_curve(yte,p); ax[0].plot(f,t,label=n)
    a,b=calibration_curve(yte,p,n_bins=8,strategy='quantile'); ax[1].plot(b,a,'o-',label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set(xlabel='FPR',ylabel='TPR',title='ROC (held-out people)'); ax[0].legend()
ax[1].plot([0,.6],[0,.6],'k--'); ax[1].set(xlabel='predicted',ylabel='observed',title='Calibration'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. What drives the prediction, and is it practically useful?

In [ ]:
m=fitted['grad. boosting']
pi=permutation_importance(m,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=0)
s=pd.Series(pi.importances_mean,Xte.columns).sort_values(); sd=pd.Series(pi.importances_std,Xte.columns)[s.index]
s.plot.barh(xerr=sd,figsize=(6,4),title='Permutation importance (drop in test AUC)'); plt.show()
lrm=fitted['logistic']; names=lrm[0].get_feature_names_out(); co=pd.Series(lrm[-1].coef_[0],names)
print('logistic coefficients (per 1 SD, numeric features):'); print(co[co.index.str.startswith('n__')].sort_values().round(3))

In [ ]:
# Lift: flag the top-k% highest-risk stays
p=pt['grad. boosting']; base=yte.mean(); r=[]
for k in (.05,.10,.20,.30):
    top=np.argsort(-p)[:int(k*len(p))]; r.append(dict(flagged=f'{k:.0%}',precision=yte[top].mean(),lift=yte[top].mean()/base,recall=yte[top].sum()/yte.sum()))
pd.DataFrame(r).round(3)

## 6. Exploratory: does a person's own history help?
Readmission is much more clustered within people than chance (section 1), so some persistent person-level risk exists that the stay-level features miss. A feature built from the **earlier** stays of the same person (`visit_number` is contiguous, 1..n) is leak-free *provided the outcome of stay k-1 is known when stay k is admitted*. We don't have dates, so this is an assumption to confirm: if the readmission flag of stay k-1 is defined by stay k itself being a readmission, this would be target leakage in deployment. Treat this as a what-if, not the headline result.

In [ ]:
h=df.sort_values(['person_key','visit_number']).copy()
grp=h.groupby('person_key').readmit_30d
h['prior_n']=h.visit_number-1
h['prior_rate']=(grp.cumsum()-h.readmit_30d)/h.prior_n.replace(0,np.nan)   # mean over strictly earlier stays
h['prior_rate']=h.prior_rate.fillna(df.readmit_30d.mean())                   # first stay: no history
h=h.loc[df.index]                                                            # restore original row order
X2=X.assign(prior_rate=h.prior_rate.values)
res={}
for name,XX in {'stay features only':X,'+ prior-stay readmit rate':X2}.items():
    p=cross_val_predict(hgb,XX.iloc[tr],ytr,groups=gtr,cv=cv,method='predict_proba')[:,1]
    mm=hgb.fit(XX.iloc[tr],ytr); pt_=mm.predict_proba(XX.iloc[te])[:,1]
    res[name]=dict(cv_AUC=roc_auc_score(ytr,p),test_AUC=roc_auc_score(yte,pt_),test_PR_AUC=average_precision_score(yte,pt_))
pd.DataFrame(res).T.round(3)

## 7. Conclusion
(Numbers from the run with `random_state=0`; re-running reproduces them.)

* **Stay-level features give only weak prediction.** On people never seen in training, AUC is about 0.63 (95% CI roughly 0.60–0.67) for both logistic regression and gradient boosting, versus 0.5 for chance. PR-AUC is about 0.31 against a 20% base rate. The signal is real but modest.
* **What carries it:** length of stay, number of medications at discharge and lab A (higher → more readmission). `lab_b`, BMI, sex, age, visit number and postcode add essentially nothing. The simpler logistic model is as good as boosting, and one-hot postcode slightly hurts (overfitting), so there is no evidence of non-linear structure worth the complexity.
* **Practical use:** flagging the top 10% highest-risk stays catches readmissions at about 37% precision (1.9× base rate) but only ~19% recall. That is useful for prioritising limited follow-up, not for reliably identifying who will be readmitted.
* **Strong person-level clustering:** readmission varies between people far more than chance (SD 0.27 vs ~0.14). Adding the readmission rate over a person's *earlier* stays raises held-out AUC to about 0.82. This is the largest gain available, **but** it depends on that history being known at admission time. Confirm how `readmit_30d` is defined and how far apart stays are before relying on it.
* **Caveats:** split by person throughout (a random row split would leak person identity and inflate scores); no dates, so no temporal validation; single dataset, no external validation; observational association, not causal.